# Attention Mechanisms: MHA, MQA, GQA, and MLA

> **Difficulty:** Intermediate | **Time:** ~50 min

The choice of attention mechanism is the single biggest lever for KV-cache memory reduction in LLM serving. This notebook walks through the evolution from Multi-Head Attention (MHA) to Multi-Latent Attention (MLA), showing exactly how each variant trades off memory, quality, and complexity.

**What you'll learn:**
- How MHA computes attention with separate K/V per head
- How MQA shares a single K/V across all heads (Falcon)
- How GQA groups heads to balance memory and quality (LLaMA-2/3)
- How MLA compresses K/V into a low-rank latent (DeepSeek-V2)
- Quantitative KV-cache comparison across real model configs
- Other variants: Linear Attention, Sliding Window, Sparse Attention
- How each variant interacts with Tensor Parallelism (split vs replicate KV heads)

**Prerequisites:** [00-kv-cache](00-kv-cache.ipynb) (understand what KV-cache is and why it matters), [02-tensor-parallelism](../../02-tensor-parallelism.ipynb) (for Section 7)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import sys, os

sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))

from mp_tutorial.inference import (
    calc_kv_cache_size,
    attention_forward_sim,
)
from mp_tutorial.inference_viz import (
    draw_attention_head_layout,
    draw_kv_cache_comparison_bar,
    draw_mla_projection_flow,
    draw_mha_vs_mqa_vs_gqa,
)
from mp_tutorial.formatting import info_box, comparison_table

import warnings
warnings.filterwarnings('ignore')

from mp_tutorial.fonts import configure_cjk_fonts
configure_cjk_fonts()

torch.manual_seed(42)
print('Ready!')

## 1. Multi-Head Attention (MHA) — The Baseline

Standard multi-head attention (Vaswani et al., 2017) projects the input into **separate Q, K, V for each head**. Each head attends independently, and the results are concatenated.

$$\text{head}_i = \text{Softmax}\left(\frac{Q_i K_i^T}{\sqrt{d_k}}\right) V_i$$

$$\text{MHA}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

During autoregressive decoding, K and V for all past tokens are cached. The KV-cache stores **2 tensors (K and V) × n_heads × head_dim** values per token per layer.

In [ ]:
# Toy MHA: 4 heads, d_model=16, seq_len=4
batch, seq_len, d_model = 1, 4, 16
n_heads = 4
head_dim = d_model // n_heads  # 4

x = torch.randn(batch, seq_len, d_model)

result = attention_forward_sim("mha", x, n_heads=n_heads, n_kv_heads=n_heads, head_dim=head_dim)

print("=== MHA Tensor Shapes ===")
print(f"Input:    {list(x.shape)}           (batch, seq_len, d_model)")
print(f"Q:        {list(result['q'].shape)}        (batch, n_heads, seq_len, head_dim)")
print(f"K:        {list(result['k'].shape)}        (batch, n_kv_heads, seq_len, head_dim)")
print(f"V:        {list(result['v'].shape)}        (batch, n_kv_heads, seq_len, head_dim)")
print(f"Scores:   {list(result['scores'].shape)}      (batch, n_heads, seq_len, seq_len)")
print(f"Output:   {list(result['output'].shape)}       (batch, seq_len, n_heads * head_dim)")
print()
print(f"KV cache per token: 2 × {n_heads} heads × {head_dim} dim = {2 * n_heads * head_dim} values")

In [ ]:
# Visualize MHA head layout
draw_attention_head_layout("mha", n_q_heads=4, n_kv_heads=4, head_dim=4)
plt.show()

In [ ]:
# KV cache cost at scale: LLaMA-2-70B config with MHA
mha_cache = calc_kv_cache_size(
    variant="mha", n_heads=64, n_kv_heads=64,
    head_dim=128, seq_len=4096, n_layers=80
)
print(f"LLaMA-2-70B (if MHA): KV-cache for 4096 tokens = {mha_cache / 1024**3:.2f} GB")
print(f"That's per request! A batch of 32 would need {32 * mha_cache / 1024**3:.1f} GB just for KV-cache.")

**Key insight:** MHA's KV-cache grows as `2 × n_layers × n_heads × head_dim × seq_len`. For large models with many heads, this becomes the dominant memory cost during serving — often exceeding the model weights themselves at long sequence lengths.

The next three variants all attack this problem by **reducing what gets stored in the KV-cache**.

## 2. Multi-Query Attention (MQA) — Share K/V Across All Heads

Multi-Query Attention (Shazeer, 2019) uses the simplest possible reduction: **all query heads share a single K and a single V**.

$$Q_i = X W_i^Q \quad (\text{separate per head})$$
$$K = X W^K, \quad V = X W^V \quad (\text{shared across all heads})$$

This reduces the KV-cache by a factor of `n_heads` — from 64 K/V pairs down to just 1. The K/V projection matrices are also smaller, reducing parameter count slightly.

**Used by:** Falcon-40B, PaLM, StarCoder

In [ ]:
# Toy MQA: 4 query heads, 1 KV head
result_mqa = attention_forward_sim("mqa", x, n_heads=4, n_kv_heads=1, head_dim=head_dim)

print("=== MQA Tensor Shapes ===")
print(f"Q:        {list(result_mqa['q'].shape)}        (batch, n_heads, seq_len, head_dim)  — still separate per head")
print(f"K:        {list(result_mqa['k'].shape)}        (batch, 1, seq_len, head_dim)  — SINGLE shared K")
print(f"V:        {list(result_mqa['v'].shape)}        (batch, 1, seq_len, head_dim)  — SINGLE shared V")
print(f"Output:   {list(result_mqa['output'].shape)}       (batch, seq_len, d_model)")
print()
print(f"KV cache per token: 2 × 1 head × {head_dim} dim = {2 * 1 * head_dim} values")
print(f"Reduction vs MHA: {n_heads}× fewer KV cache values!")

In [ ]:
# Visualize: how broadcasting works in MQA
# The single K/V is broadcast (repeated) to match each query head
draw_attention_head_layout("mqa", n_q_heads=4, n_kv_heads=1, head_dim=4)
plt.show()

In [ ]:
# MQA cache savings at scale: Falcon-40B config
mqa_cache = calc_kv_cache_size(
    variant="mqa", n_heads=64, n_kv_heads=1,
    head_dim=128, seq_len=4096, n_layers=60
)
print(f"Falcon-40B (MQA): KV-cache for 4096 tokens = {mqa_cache / 1024**3:.4f} GB")
print(f"vs MHA with same config: {calc_kv_cache_size('mha', 64, 64, 128, 4096, 60) / 1024**3:.2f} GB")
print(f"Reduction: {64}×")

**Trade-off:** MQA gives massive memory savings but can hurt model quality. With only one K/V representation, all heads are forced to attend over the same key-value space — they lose the ability to specialize. In practice, models trained from scratch with MQA work well (Falcon, PaLM), but uptraining an existing MHA model to MQA can degrade quality.

This motivates GQA: a middle ground between MHA and MQA.

## 3. Grouped-Query Attention (GQA) — The Middle Ground

Grouped-Query Attention (Ainslie et al., 2023) partitions query heads into **groups**, where each group shares one K/V head. With `n_kv_heads` groups:

- `n_kv_heads = n_heads` → equivalent to **MHA** (every head has its own K/V)
- `n_kv_heads = 1` → equivalent to **MQA** (all heads share one K/V)
- `1 < n_kv_heads < n_heads` → **GQA** (grouped sharing)

$$\text{group}(i) = \lfloor i \cdot n_{kv} / n_q \rfloor$$

**Used by:** LLaMA-2-70B (8 KV heads), LLaMA-3 (8 KV heads), Mistral (8 KV heads)

In [ ]:
# Toy GQA: 4 query heads, 2 KV heads (groups of 2)
result_gqa = attention_forward_sim("gqa", x, n_heads=4, n_kv_heads=2, head_dim=head_dim)

print("=== GQA Tensor Shapes (4 Q heads, 2 KV heads) ===")
print(f"Q:        {list(result_gqa['q'].shape)}        (batch, n_q_heads, seq_len, head_dim)")
print(f"K:        {list(result_gqa['k'].shape)}        (batch, n_kv_heads, seq_len, head_dim)  — 2 KV heads")
print(f"V:        {list(result_gqa['v'].shape)}        (batch, n_kv_heads, seq_len, head_dim)")
print(f"Output:   {list(result_gqa['output'].shape)}       (batch, seq_len, d_model)")
print()
print(f"Q heads 0,1 share KV head 0 | Q heads 2,3 share KV head 1")
print(f"KV cache per token: 2 × 2 heads × {head_dim} dim = {2 * 2 * head_dim} values")
print(f"Reduction vs MHA: {n_heads // 2}×")

In [ ]:
# Visualize GQA grouping
draw_attention_head_layout("gqa", n_q_heads=8, n_kv_heads=2, head_dim=128)
plt.show()

In [ ]:
# Verify: GQA generalizes to MHA and MQA
print("GQA as generalization:")
print()

# When n_kv_heads = n_heads -> MHA
result_gqa_as_mha = attention_forward_sim("gqa", x, n_heads=4, n_kv_heads=4, head_dim=head_dim)
result_pure_mha = attention_forward_sim("mha", x, n_heads=4, n_kv_heads=4, head_dim=head_dim)
match_mha = torch.allclose(result_gqa_as_mha['output'], result_pure_mha['output'], atol=1e-6)
print(f"GQA(n_kv=n_q=4) == MHA: {match_mha}")

# When n_kv_heads = 1 -> MQA
result_gqa_as_mqa = attention_forward_sim("gqa", x, n_heads=4, n_kv_heads=1, head_dim=head_dim)
result_pure_mqa = attention_forward_sim("mqa", x, n_heads=4, n_kv_heads=1, head_dim=head_dim)
match_mqa = torch.allclose(result_gqa_as_mqa['output'], result_pure_mqa['output'], atol=1e-6)
print(f"GQA(n_kv=1)    == MQA: {match_mqa}")

In [ ]:
# Overview of all three so far using existing viz
draw_mha_vs_mqa_vs_gqa(n_q_heads=8, head_dim=128)
plt.show()

**Key takeaway:** GQA gives you a tunable knob between memory and quality. LLaMA-2-70B uses 64 query heads but only 8 KV heads — an 8× KV-cache reduction while preserving most of MHA's quality. This has become the industry default for large models.

But can we do even better? What if instead of reducing the *number* of KV heads, we compress the *representation* itself?

## 4. Multi-Latent Attention (MLA) — Low-Rank KV Compression

Multi-Latent Attention (DeepSeek-AI, 2024) takes a fundamentally different approach. Instead of sharing KV heads across query groups, MLA **compresses** the KV representation into a low-rank latent space before caching.

The key idea:
1. **Down-project** the input into a compressed latent: $c = X W_{\text{down}}$ where $c \in \mathbb{R}^{d_{\text{compressed}}}$
2. **Cache** only the compressed latent $c$ (much smaller than separate K, V)
3. **Up-project** back to K and V when computing attention: $K = c W_{\text{up}}^K$, $V = c W_{\text{up}}^V$

The cache stores a single vector of dimension $d_{\text{compressed}}$ per token, instead of separate K and V of dimension $n_{\text{heads}} \times d_{\text{head}}$ each.

**Used by:** DeepSeek-V2, DeepSeek-V3

In [ ]:
# Toy MLA: compress d_model=16 down to d_compressed=6
d_compressed = 6

result_mla = attention_forward_sim(
    "mla", x, n_heads=4, n_kv_heads=4, head_dim=head_dim, d_compressed=d_compressed
)

print("=== MLA Tensor Shapes ===")
print(f"Q:              {list(result_mla['q'].shape)}        (batch, n_heads, seq_len, head_dim)")
print(f"Compressed KV:  {list(result_mla['compressed_kv'].shape)}       (batch, seq_len, d_compressed) ← THIS IS CACHED")
print(f"K (from decomp):{list(result_mla['k'].shape)}        (batch, n_heads, seq_len, head_dim) — reconstructed")
print(f"V (from decomp):{list(result_mla['v'].shape)}        (batch, n_heads, seq_len, head_dim) — reconstructed")
print(f"Output:         {list(result_mla['output'].shape)}       (batch, seq_len, d_model)")
print()
mha_cache_per_token = 2 * n_heads * head_dim
mla_cache_per_token = d_compressed
print(f"MHA caches: 2 × {n_heads} × {head_dim} = {mha_cache_per_token} values/token")
print(f"MLA caches: {d_compressed} values/token (compressed latent)")
print(f"Compression ratio: {mha_cache_per_token / mla_cache_per_token:.1f}×")

In [ ]:
# Visualize the MLA pipeline
draw_mla_projection_flow(d_model=512, d_compressed=64, n_heads=8, head_dim=64)
plt.show()

In [ ]:
# MLA cache savings at scale: DeepSeek-V2 config
# DeepSeek-V2: 128 heads, head_dim=128, 60 layers, d_compressed=512
mla_cache = calc_kv_cache_size(
    variant="mla", n_heads=128, n_kv_heads=128,
    head_dim=128, seq_len=4096, n_layers=60, d_compressed=512
)
mha_equivalent = calc_kv_cache_size(
    variant="mha", n_heads=128, n_kv_heads=128,
    head_dim=128, seq_len=4096, n_layers=60
)
print(f"DeepSeek-V2 (MLA, d_c=512): KV-cache = {mla_cache / 1024**3:.4f} GB")
print(f"Same model with MHA:        KV-cache = {mha_equivalent / 1024**3:.2f} GB")
print(f"Reduction: {mha_equivalent / mla_cache:.0f}×")

**Trade-off:** MLA achieves extreme compression but adds computational cost — the up-projection from compressed latent to K/V must happen at every attention step. This is a compute-for-memory trade: extra matrix multiplications in exchange for dramatically smaller KV-cache. For memory-bound inference workloads (long sequences, large batches), this trade is overwhelmingly favorable.

## 5. Comparison: Putting It All Together

Let's compare all four variants side-by-side using real model configurations.

In [ ]:
# Side-by-side KV cache comparison with real model configs
configs = [
    {"name": "MHA\n(hypothetical 70B)", "variant": "mha",
     "n_heads": 64, "n_kv_heads": 64, "head_dim": 128, "n_layers": 80},
    {"name": "MQA\n(Falcon-40B)", "variant": "mqa",
     "n_heads": 64, "n_kv_heads": 1, "head_dim": 128, "n_layers": 60},
    {"name": "GQA\n(LLaMA-2-70B)", "variant": "gqa",
     "n_heads": 64, "n_kv_heads": 8, "head_dim": 128, "n_layers": 80},
    {"name": "MLA\n(DeepSeek-V2)", "variant": "mla",
     "n_heads": 128, "n_kv_heads": 128, "head_dim": 128, "n_layers": 60,
     "d_compressed": 512},
]

draw_kv_cache_comparison_bar(configs)
plt.show()

In [ ]:
# Summary table
print("┌──────────┬──────────────┬───────────────┬──────────────────────────────────────┐")
print("│ Variant  │ KV Heads     │ Cache / Token │ Key Trade-off                        │")
print("├──────────┼──────────────┼───────────────┼──────────────────────────────────────┤")
print("│ MHA      │ n_heads      │ 2·h·d         │ Full quality, full memory cost        │")
print("│ MQA      │ 1            │ 2·d           │ Max savings, some quality loss        │")
print("│ GQA      │ g (tunable)  │ 2·g·d         │ Balanced — industry default           │")
print("│ MLA      │ n/a (latent) │ d_compressed  │ Max compression, extra compute cost   │")
print("└──────────┴──────────────┴───────────────┴──────────────────────────────────────┘")
print()
print("h = n_heads, d = head_dim, g = n_kv_groups")
print()
print("Evolution: MHA (2017) → MQA (2019) → GQA (2023) → MLA (2024)")
print("Each step trades something to reduce KV-cache memory.")

## 6. Other Attention Variants: Beyond Softmax

The MHA→MQA→GQA→MLA family all use standard softmax attention — they reduce **what gets cached**, not how attention is computed. A separate line of research attacks the O(n²) complexity of attention itself.

### Linear Attention

Standard attention computes $\text{Softmax}(QK^T)V$, which materializes an $n \times n$ attention matrix. Linear Attention (Katharopoulos et al., 2020) replaces softmax with a kernel function $\phi$:

$$\text{Standard: } \quad O = \text{Softmax}(QK^T) \cdot V \qquad \mathcal{O}(n^2 d)$$
$$\text{Linear: } \quad O = \phi(Q) \cdot (\phi(K)^T V) \qquad \mathcal{O}(n d^2)$$

The trick is associativity: by computing $\phi(K)^T V$ first (a $d \times d$ matrix), we avoid the $n \times n$ product entirely. This makes attention **O(n)** in sequence length, but the constant factor $d^2$ means it only wins for very long sequences.

**Descendants:** RetNet (multi-scale retention), RWKV (linear recurrence as attention), Mamba/S4 (state-space models — not technically "attention" but solve the same problem). These architectures can process sequences in O(n) time and maintain a **fixed-size state** instead of a growing KV-cache.

### Sliding Window Attention

Sliding Window Attention (Beltagy et al., 2020; used in Mistral, Gemma-2) restricts each token to attend only within a fixed window of size $w$:

$$\text{Attn}(q_i) = \text{Softmax}(q_i \cdot K[i-w:i]^T) \cdot V[i-w:i]$$

This **caps KV-cache** at $w$ tokens regardless of total sequence length — a hard upper bound. Information beyond the window propagates indirectly through stacked layers: with $L$ layers and window $w$, the effective receptive field is $L \times w$ tokens.

**Used by:** Mistral-7B ($w = 4096$), Gemma-2 (alternating global + sliding layers)

### Sparse Attention

Longformer and BigBird combine **local windows** (each token sees neighbors) with **global tokens** (a few tokens see everything). This achieves O(n) complexity while preserving long-range dependencies through the global tokens.

**Key insight:** These approaches are largely **orthogonal** to MQA/GQA/MLA. You can combine sliding window with GQA (Mistral does exactly this) or with MLA. They reduce different costs: head-sharing reduces KV-cache *size per token*, while sparse/linear methods reduce the *number of tokens* that participate in attention.

## 7. Attention Variants and Tensor Parallelism

When serving large models across multiple GPUs, Tensor Parallelism (TP) splits each layer across devices. For attention, the standard Megatron-LM approach is **column-parallel Q/K/V projections** (split by heads) followed by a **row-parallel output projection** with a single AllReduce. But the different attention variants interact with TP very differently.

### MHA + TP: The Clean Case

With `n_heads` independent Q, K, V heads, TP is straightforward:
- Split: each GPU gets `n_heads / tp_size` complete heads (Q, K, and V)
- Compute: each GPU runs attention on its local heads — **zero communication**
- Merge: row-parallel $W_O$ followed by AllReduce

This works perfectly because heads are independent. With 64 heads on 8 GPUs → 8 heads/GPU, cleanly divisible.

### MQA + TP: The Replication Problem

MQA has only **1 KV head** but `n_heads` Q heads. You can't split 1 head across GPUs. Two options:

1. **Replicate KV** on every GPU (standard approach): each GPU holds the full single K/V head and its subset of Q heads. The KV is tiny (1 head), so replication overhead is negligible.
2. **Broadcast KV** from one GPU: compute KV on rank 0, then broadcast. Adds a communication step but avoids redundant computation.

In practice, option 1 is used because the single KV head is so small that replicating it is cheaper than the broadcast latency.

### GQA + TP: It Depends on the Ratio

The key question: **is `n_kv_heads` divisible by `tp_size`?**

| Scenario | Example | Strategy |
|----------|---------|----------|
| `n_kv_heads >= tp_size` and divisible | 8 KV heads on 8 GPUs | Split: 1 KV head/GPU |
| `n_kv_heads >= tp_size` but not divisible | 8 KV heads on 6 GPUs | Can't split evenly — avoid this config |
| `n_kv_heads < tp_size` | 8 KV heads on 16 GPUs | **Replicate**: each KV head copied to `tp_size / n_kv_heads` GPUs |

LLaMA-2-70B with 8 KV heads on 8 GPUs is the sweet spot: exactly 1 KV head per GPU, with 8 Q heads per GPU sharing it locally.

**Practical tip:** When choosing `n_kv_heads` for a new model, pick a value that divides your target `tp_size`. This is why you see 8 KV heads so often — it divides 8, 16, 32, and 64 GPUs cleanly.

### MLA + TP: Split the Up-Projection

MLA's TP strategy differs fundamentally because there are no KV "heads" to split:

1. **Down-projection** $W_{\text{down}}$: produces the shared compressed latent $c$. This must be computed fully (or replicated), since all heads need it.
2. **Compressed latent**: replicated on all GPUs (it's small: `d_compressed` per token).
3. **Up-projections** $W_{\text{up}}^K$, $W_{\text{up}}^V$: these are column-parallel — each GPU up-projects to its subset of heads.
4. **Attention + output projection**: same as MHA from here — each GPU has its local heads.

The compressed latent replication is MLA's TP advantage: instead of replicating `2 × n_heads × head_dim` per token (MHA), you replicate only `d_compressed` — the same compression that saves cache memory also reduces TP communication.

In [ ]:
# TP strategies at a glance
print("┌──────────┬────────────────────┬──────────────────────┬─────────────────────────────┐")
print("│ Variant  │ Q Split            │ KV Split             │ TP Gotcha                   │")
print("├──────────┼────────────────────┼──────────────────────┼─────────────────────────────┤")
print("│ MHA      │ n_heads / tp_size  │ n_heads / tp_size    │ None — clean split           │")
print("│ MQA      │ n_heads / tp_size  │ Replicate (1 head)   │ KV replicated, not split     │")
print("│ GQA      │ n_heads / tp_size  │ n_kv / tp_size *     │ * Replicate if n_kv < tp     │")
print("│ MLA      │ n_heads / tp_size  │ Replicate latent     │ Up-proj is column-parallel   │")
print("└──────────┴────────────────────┴──────────────────────┴─────────────────────────────┘")
print()

# Concrete example: LLaMA-2-70B GQA on different TP sizes
n_q, n_kv = 64, 8
for tp in [1, 2, 4, 8, 16]:
    q_per_gpu = n_q // tp
    if n_kv >= tp:
        kv_per_gpu = n_kv // tp
        kv_strategy = f"{kv_per_gpu} KV heads/GPU (split)"
    else:
        replicas = tp // n_kv
        kv_strategy = f"each KV head on {replicas} GPUs (replicate)"
    print(f"  TP={tp:2d}: {q_per_gpu:2d} Q heads/GPU, {kv_strategy}")

## Summary

| Variant | Paper | Cache Formula | Real Example | Relative Cache |
|---------|-------|---------------|--------------|----------------|
| **MHA** | Vaswani 2017 | `2 · n_heads · d_head` per token/layer | — | 1× (baseline) |
| **MQA** | Shazeer 2019 | `2 · d_head` per token/layer | Falcon-40B, PaLM | 1/n_heads × |
| **GQA** | Ainslie 2023 | `2 · n_kv_heads · d_head` per token/layer | LLaMA-2-70B (8 KV) | 1/group_size × |
| **MLA** | DeepSeek 2024 | `d_compressed` per token/layer | DeepSeek-V2 (512) | d_c / (2·h·d) × |

**Key takeaways:**
- KV-cache memory is the bottleneck for LLM serving at scale
- GQA is the current industry standard (LLaMA, Mistral, Gemma) — simple, effective, tunable
- MLA pushes compression further via low-rank projections, at the cost of extra compute
- The choice depends on your deployment constraints: memory-bound → favor MLA/MQA, compute-bound → MHA/GQA may suffice

**Further reading:**
- [00-kv-cache](00-kv-cache.ipynb) — What is KV-cache and why it matters
- [01-flash-attention](01-flash-attention.ipynb) — Efficient attention *computation* (orthogonal to head-sharing)
- [03-paged-attention](03-paged-attention.ipynb) — Efficient attention *memory management*
- [08-serving-architecture](08-serving-architecture.ipynb) — How these choices affect serving system design